# Парсинг данных ТОП 250-Фильмов

### Подключение библиотек

In [1]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import time
import re

### URL ссылка сайта и получение информации

In [2]:
url = 'https://www.kinoafisha.info/rating/movies/imdb/' # страница со всеми статьями 
page = requests.get(url)

In [3]:
page.status_code

200

In [4]:
soup = bs(page.text, 'html.parser')

In [5]:
page.text

'<!DOCTYPE html>\n<html lang="ru">\n<head>\n\t<meta name="revisit-after" content="1 day">\n\t<meta charset="utf-8">\n\t<meta http-equiv="X-UA-Compatible" content="IE=edge">\n\t\t<title>Топ-250 фильмов по версии imdb - рейтинг фильмов | Киноафиша</title>\n\n    <meta name="viewport" content="width=device-width, initial-scale=1">\n    <script>if(screen.width>=759) document.currentScript.previousElementSibling.setAttribute(\'content\', \'width=990, initial-scale=1\');</script>\n\t<link rel="preconnect" href="https://cdn.advertronic.io" crossorigin>\n\n\t<style>\n        :root{\n            --ads-placeholder:\'Реклама\';\n        }\n        :root{--desktop:759px;--small-columns:1280px;--normal-columns:1325px;--wide-grid:1365px;--normal-column:990;--normal-column-screen:1010;--wide-column:1030;--wide-column-screen:1050;--column-width-small:930px;--column-width-normal:970px;--icon-preloader:url("data:image/svg+xml;charset=utf8,%3C?xml version=\'1.0\' encoding=\'UTF-8\' standalone=\'no\'?%3E%

In [6]:
soup

<!DOCTYPE html>

<html lang="ru">
<head>
<meta content="1 day" name="revisit-after"/>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<title>Топ-250 фильмов по версии imdb - рейтинг фильмов | Киноафиша</title>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<script>if(screen.width>=759) document.currentScript.previousElementSibling.setAttribute('content', 'width=990, initial-scale=1');</script>
<link crossorigin="" href="https://cdn.advertronic.io" rel="preconnect"/>
<style>
        :root{
            --ads-placeholder:'Реклама';
        }
        :root{--desktop:759px;--small-columns:1280px;--normal-columns:1325px;--wide-grid:1365px;--normal-column:990;--normal-column-screen:1010;--wide-column:1030;--wide-column-screen:1050;--column-width-small:930px;--column-width-normal:970px;--icon-preloader:url("data:image/svg+xml;charset=utf8,%3C?xml version='1.0' encoding='UTF-8' standalone='no'?%3E%3Csvg xmlns:svg='http://www.w3.org/2000/svg'

# Загрузка главной страницы

In [7]:
!pip install curl_cffi

In [1]:
import re
import time
import csv
import os

from bs4 import BeautifulSoup as bs
from curl_cffi import requests
from curl_cffi.requests.exceptions import RequestException


base_url = "https://www.kinoafisha.info"
rating_url = "https://www.kinoafisha.info/rating/movies/imdb/"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,'
              'image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.kinoafisha.info/',
}


# ============================================================
# ЗАГРУЗКА ГЛАВНОЙ СТРАНИЦЫ И ОБЪЯВЛЕНИЕ rating_blocks
# ============================================================
print("Загружаем главную страницу рейтинга...")
try:
    response = requests.get(
        rating_url,
        headers=headers,
        timeout=30,
        impersonate="chrome"
    )
    response.raise_for_status()
except RequestException as e:
    print(f"Ошибка загрузки: {e}")
    exit()

with open('debug_page.html', 'w', encoding='utf-8') as f:
    f.write(response.text)

soup = bs(response.text, 'html.parser')

# вот тут rating_blocks и появляется
rating_blocks = soup.find_all(
    string=lambda t: t and re.match(r'^\s*\d+[.,]\d+\s*$', t)
)
print(f"Найдено rating_blocks: {len(rating_blocks)}")

if not rating_blocks:
    print("⚠ rating_blocks пуст — смотри debug_page.html")
    exit()


# ============================================================
# ДАЛЬШЕ — ТВОЙ СТАРЫЙ КОД (без изменений)
# ============================================================
result_list = {
    'position': [], 'title': [], 'rating': [],
    'genre': [], 'year': [], 'country': [],
    'describe': [], 'film_url': []
}

for i, rating_el in enumerate(rating_blocks[:250]):
    try:
        # 1. Рейтинг
        rating = rating_el.strip() if isinstance(rating_el, str) else rating_el.get_text(strip=True)

        # 2. Ссылка
        link_el = rating_el.find_next('a', href=lambda h: h and '/movies/' in h and '/rating/' not in h)

        if not link_el:
            continue

        film_href = link_el.get('href')
        film_url = film_href if film_href.startswith('http') else f"{base_url}{film_href}"

        # 3. Название
        title = link_el.get_text(strip=True)

        # 4. Позиция
        position = 'N/A'

        parent = link_el.find_parent()
        if parent:
            text_content = parent.get_text(separator=' ', strip=True)
            pos_match = re.match(r'^(\d+)', text_content)
            if pos_match:
                position = pos_match.group(1)

        if position == 'N/A':
            prev_el = rating_el.find_previous(string=lambda text: text and re.match(r'^\s*\d+\s*$', text.strip()))
            if prev_el:
                position = prev_el.strip()
            else:
                position = str(i + 1)

        # 5. Мета-инфо (Жанр, Год, Страна)
        genre, year, country = 'N/A', 'N/A', 'N/A'

        meta_text = ""

        if parent:
            meta_text = parent.get_text(separator=' ', strip=True)

        if not meta_text or len(meta_text) < 5:
            next_sib = link_el.find_next_sibling()
            if next_sib:
                meta_text = next_sib.get_text(strip=True)
            else:
                next_text = rating_el.find_next(string=lambda t: t and re.search(r'(19|20)\d{2}', t))
                if next_text:
                    meta_text = next_text

        if meta_text:
            year_match = re.search(r'(19|20)\d{2}', meta_text)
            if year_match:
                year = year_match.group(0)
                parts = meta_text.split(year)
                if len(parts) > 0:
                    genre_part = parts[0].strip().strip(',').strip()
                    if genre_part and len(genre_part) > 2 and 'Рейтинг' not in genre_part:
                        genre = genre_part.split(',')[-1].strip()

                if len(parts) > 1:
                    country_part = parts[1].strip().strip(',').strip()
                    if country_part and len(country_part) > 2:
                        country = country_part.split(',')[0].strip()

        # 6. Описание
        description = 'N/A'
        try:
            film_resp = requests.get(
                film_url,
                headers=headers,
                timeout=15,
                impersonate="chrome"
            )
            if film_resp.status_code == 200:
                film_soup = bs(film_resp.text, 'html.parser')

                found_desc = False
                for tag in film_soup.find_all(['p', 'div', 'span']):
                    text = tag.get_text(strip=True)
                    if 150 < len(text) < 1000:
                        if any(x in text.lower() for x in ['сюжет', 'история', 'фильм рассказывает']):
                            description = text
                            found_desc = True
                            break

                if not found_desc:
                    for tag in film_soup.find_all('p'):
                        text = tag.get_text(strip=True)
                        if 200 < len(text) < 800:
                            description = text
                            break

        except Exception as e:
            print(f"Не удалось загрузить страницу {e}")

        # 7. Сохранение
        result_list['position'].append(position)
        result_list['title'].append(title)
        result_list['rating'].append(rating)
        result_list['genre'].append(genre)
        result_list['year'].append(year)
        result_list['country'].append(country)
        result_list['describe'].append(description)
        result_list['film_url'].append(film_url)

        print(f"{position}. {title} ({year}) — {rating}")

        time.sleep(0.1)

    except Exception as e:
        print(f"Ошибка обработки фильма #{i+1}: {e}")
        continue


# ============================================================
# СОХРАНЕНИЕ В CSV
# ============================================================
output_file = "kinoafisha_rating.csv"
with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(result_list.keys())
    for i in range(len(result_list['position'])):
        writer.writerow([result_list[key][i] for key in result_list])

print(f"\nГотово! Сохранено {len(result_list['position'])} фильмов в {output_file}")

Загружаем главную страницу рейтинга...
Найдено rating_blocks: 260
1. Побег из Шоушенка (1994) — 9.3
2. Крестный отец (1972) — 9.2
3. Темный рыцарь (2008) — 9.1
4. Крестный отец 2 (1974) — 9.0
5. Властелин Колец: Возвращение Короля (2003) — 9.0
6. 12 разгневанных мужчин (1957) — 9.0
7. Список Шиндлера (1993) — 9.0
8. Властелин Колец: Братство Кольца (2001) — 8.9
9. Криминальное чтиво (1994) — 8.8
10. Властелин Колец: Две крепости (2002) — 8.8
11. Хороший, плохой, злой (1966) — 8.8
12. Форрест Гамп (1994) — 8.8
13. Начало (2010) — 8.8
14. Бойцовский клуб (1999) — 8.8
15. Звездные войны: Эпизод V - Империя наносит ответный удар (1980) — 8.7
16. Матрица (1999) — 8.7
17. Интерстеллар (2014) — 8.7
18. Славные парни (1990) — 8.7
19. Пролетая над гнездом кукушки (1975) — 8.6
20. Семь (1995) — 8.6
Не удалось загрузить страницу Failed to perform, curl: (35) Recv failure: Connection was reset. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
21. Эта замечательная жизнь (1

### Парсинг

In [ ]:
result_list = {
    'position': [], 'title': [], 'rating': [], 
    'genre': [], 'year': [], 'country': [], 
    'describe': [], 'film_url': []
}

for i, rating_el in enumerate(rating_blocks[:250]):
    try:
        # 1. Рейтинг
        rating = rating_el.strip() if isinstance(rating_el, str) else rating_el.get_text(strip=True)
        
        # 2. Ссылка
        # Ищем ссылку на фильм. Лучше искать относительно текущего блока, но не улетать далеко
        link_el = rating_el.find_next('a', href=lambda h: h and '/movies/' in h and '/rating/' not in h)
        
        if not link_el:
            continue # Нет ссылки — нет фильма, пропускаем
            
        film_href = link_el.get('href')
        film_url = film_href if film_href.startswith('http') else f"{base_url}{film_href}"
        
        # 3. Название
        # Берем текст ИЗ ссылки. Это на 99% правильное название фильма.
        title = link_el.get_text(strip=True)
        
        # 4. Позиция
        # Пытаемся найти цифру позиции. Часто она в том же блоке, что и ссылка, или перед ней.
        # Ищем не просто строку, а элемент, который содержит цифру в начале
        position = 'N/A'
        
        # Вариант А: Позиция внутри родителя ссылки (если структура <div><span>1.</span><a>Title</a></div>)
        parent = link_el.find_parent()
        if parent:
            # Ищем текст в родителе, который начинается с цифры
            text_content = parent.get_text(separator=' ', strip=True)
            pos_match = re.match(r'^(\d+)', text_content)
            if pos_match:
                position = pos_match.group(1)
        
        if position == 'N/A':
            # Ищем предыдущие элементы, но не уходим далеко
            prev_el = rating_el.find_previous(string=lambda text: text and re.match(r'^\s*\d+\s*$', text.strip()))
            if prev_el:
                position = prev_el.strip()
            else:
                # Если совсем ничего, ставим порядковый номер из цикла + 1 (как запасной вариант)
                position = str(i + 1)

        # 5. Мета-инфо (Жанр, Год, Страна)
        genre, year, country = 'N/A', 'N/A', 'N/A'
        
        # Ищем мета-инфо рядом со ссылкой, но в пределах разумного (в том же блоке)
        # Обычно это следующий sibling или текст в родителе после ссылки
        meta_text = ""
        
        # Пробуем взять текст из родителя ссылки (там часто всё вместе)
        if parent:
            meta_text = parent.get_text(separator=' ', strip=True)
        
        # Если в родителе пусто, ищем следующего соседа, но не дальше 3 элементов
        if not meta_text or len(meta_text) < 5:
            next_sib = link_el.find_next_sibling()
            if next_sib:
                meta_text = next_sib.get_text(strip=True)
            else:
                # Запасной вариант: ищем ближайший текст с годом
                next_text = rating_el.find_next(string=lambda t: t and re.search(r'(19|20)\d{2}', t))
                if next_text:
                    meta_text = next_text

        if meta_text:
            # Парсим год
            year_match = re.search(r'(19|20)\d{2}', meta_text)
            if year_match:
                year = year_match.group(0)
                # Всё что до года - жанр, всё что после - страна
                parts = meta_text.split(year)
                if len(parts) > 0:
                    genre_part = parts[0].strip().strip(',').strip()
                    # Чистим жанр от мусора
                    if genre_part and len(genre_part) > 2 and 'Рейтинг' not in genre_part:
                        genre = genre_part.split(',')[-1].strip() # Часто жанр перед годом
                
                if len(parts) > 1:
                    country_part = parts[1].strip().strip(',').strip()
                    if country_part and len(country_part) > 2:
                        country = country_part.split(',')[0].strip()

        # 6. Описание
        description = 'N/A'
        try:
            film_resp = requests.get(film_url, headers=headers, timeout=15)
            if film_resp.status_code == 200:
                film_soup = bs(film_resp.text, 'html.parser')
                
                # Попытка найти сюжет
                found_desc = False
                # Часто описание в конкретном блоке, попробуем найти по классам или ключевым словам
                for tag in film_soup.find_all(['p', 'div', 'span']):
                    text = tag.get_text(strip=True)
                    if 150 < len(text) < 1000:
                        # Если внутри есть слова "сюжет", "история" или просто чистый текст
                        if any(x in text.lower() for x in ['сюжет', 'история', 'фильм рассказывает']):
                            description = text
                            found_desc = True
                            break
                
                # Если не нашли по ключевым словам, берем первый подходящий большой кусок
                if not found_desc:
                    for tag in film_soup.find_all('p'):
                        text = tag.get_text(strip=True)
                        if 200 < len(text) < 800:
                            description = text
                            break
                            
        except Exception as e:
            print(f"Не удалось загрузить страницу {e}")
        
        # 7. Сохранение
        result_list['position'].append(position)
        result_list['title'].append(title)
        result_list['rating'].append(rating)
        result_list['genre'].append(genre)
        result_list['year'].append(year)
        result_list['country'].append(country)
        result_list['describe'].append(description)
        result_list['film_url'].append(film_url)
        
        print(f"{position}. {title} ({year}) — {rating}")
        
        time.sleep(0.1) # Задержка
        
    except Exception as e:
        print(f"Ошибка обработки фильма #{i+1}: {e}")
        continue

In [ ]:
result_list

In [ ]:
df = pd.DataFrame(result_list)

### Вывод таблицы

In [ ]:
df.head(10)

### Сохранение в csv файл

In [ ]:
df.to_csv('top250_imdb_kinoafisha.csv', index=False, encoding='utf-8-sig')